# LeetCode 477 — Total Hamming Distance bằng Python

Link đề: https://leetcode.com/problems/total-hamming-distance/ · Độ khó: **Medium** · Chủ đề: Bit Manipulation (thao tác bit), Math (đếm tổ hợp), Array

Skeleton LeetCode:

```python
class Solution:
    def totalHammingDistance(self, nums: List[int]) -> int:
        ...
```

Ghi chú: bài này **không** phải Premium. Đây là bài kinh điển về **"tách theo từng bit"**: thay vì xét từng cặp số (`O(n^2)`), ta xét từng **vị trí bit** và đếm nhanh số cặp khác nhau tại bit đó.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: **khoảng cách Hamming** giữa hai số là số vị trí bit mà chúng khác nhau; cần cộng khoảng cách này trên **mọi cặp** `(i, j)` với `i < j`.
2. Dùng các công cụ nền: biểu diễn nhị phân (`bin`, `format`), phép dịch bit `>>`, `<<`, phép `&`, `^` (XOR), đếm bit `1` bằng `bin(x).count("1")`, và đếm số cặp bằng tích `a * b`.
3. Nhận ra bước đột phá: các bit **độc lập** nhau; tại một bit có `a` số mang bit `1` và `n - a` số mang bit `0` thì có đúng `a * (n - a)` cặp khác nhau tại bit đó, nên đáp án `= Σ_bit a * (n - a)`.
4. Cài đặt 4 cách khác nhau: **vét cạn mọi cặp `O(n^2)` → đếm theo từng bit `O(30 n)` → chia để trị theo vị trí phần tử (trái + phải + cặp chéo) → duyệt trực tuyến, cộng dồn từng số mới với số đã thấy**, và so sánh.
5. Tránh các lỗi kinh điển: cộng giá trị XOR thay vì đếm bit, dùng `a` hay `a + b` thay vì `a * b`, đếm mỗi cặp hai lần, quét thiếu bit, nhầm `x & (1 << i)` với bit `0/1`, và bỏ sót độ phức tạp `O(n^2)` của vét cạn.

## 2. Đề bài & ví dụ

**Khoảng cách Hamming** giữa hai số nguyên là số vị trí mà các bit tương ứng của chúng **khác nhau**.

Cho mảng số nguyên `nums`, trả về **tổng khoảng cách Hamming** giữa **tất cả các cặp** số trong `nums`.

**Ví dụ 1:** `nums = [4, 14, 2]` → `6`. Viết nhị phân 4 bit: `4 = 0100`, `14 = 1110`, `2 = 0010`. Khoảng cách: `d(4,14) = 2`, `d(4,2) = 2`, `d(14,2) = 2`, tổng `6`.

**Ví dụ 2:** `nums = [4, 14, 4]` → `4`. `d(4,14) = 2`, `d(4,4) = 0`, `d(14,4) = 2`, tổng `4`.

**Ràng buộc:**

- `1 <= nums.length <= 10^4`
- `0 <= nums[i] <= 10^9`
- Đáp án với input cho trước **vừa trong số nguyên 32 bit**.

Với `n = 10^4`, số cặp `n(n-1)/2 ≈ 5 × 10^7`, quá chậm cho Python nếu mỗi cặp tốn một phép đếm bit. Nhưng `10^9 < 2^30` nên mỗi số có tối đa **30 bit**: gợi ý xử lý theo từng bit.

In [1]:
import random
import time
from typing import List

examples = [([4, 14, 2], 6), ([4, 14, 4], 4)]
for nums, ans in examples:
    print(f"nums = {nums}  -> {ans}")

nums = [4, 14, 2]  -> 6
nums = [4, 14, 4]  -> 4


## 3. Kiến thức nền

### 3.1. Biểu diễn nhị phân: `bin` và `format`

`bin(x)` cho chuỗi nhị phân có tiền tố `0b`; `format(x, "04b")` cho chuỗi nhị phân **đủ 4 ký tự**, đệm số `0` bên trái, rất tiện để đối chiếu các số cùng độ rộng.

In [2]:
for x in (4, 14, 2):
    print(x, bin(x), format(x, "04b"))       # 4 0b100 0100 ...

4 0b100 0100
14 0b1110 1110
2 0b10 0010


### 3.2. Phép `>>` (dịch phải) và `& 1` (lấy bit thấp nhất)

`x >> i` dịch các bit của `x` sang phải `i` vị trí (bỏ `i` bit thấp). `& 1` giữ bit cuối cùng. Vậy `x >> i & 1` là **bit thứ `i`** của `x` (đếm từ `0`, bit thấp nhất). Chú ý: `>>` có độ ưu tiên **cao hơn** `&` nên không cần ngoặc.

In [3]:
x = 14                     # 1110
for i in range(4):
    print(f"bit {i} cua {x} ({format(x, '04b')}) = {x >> i & 1}")
# bit 0 = 0, bit 1 = 1, bit 2 = 1, bit 3 = 1

bit 0 cua 14 (1110) = 0
bit 1 cua 14 (1110) = 1
bit 2 cua 14 (1110) = 1
bit 3 cua 14 (1110) = 1


### 3.3. XOR `^` và đếm bit `1`

`a ^ b` có bit `1` ở đúng những vị trí **a và b khác nhau**. Vì vậy khoảng cách Hamming của `a` và `b` là **số bit `1` của `a ^ b`**, đếm bằng `bin(a ^ b).count("1")`.

In [4]:
a, b = 4, 14
print(format(a, "04b"), format(b, "04b"), "->", format(a ^ b, "04b"))   # 0100 1110 -> 1010
print("Hamming(4, 14) =", bin(a ^ b).count("1"))                        # 2
print("Hamming(4, 4)  =", bin(4 ^ 4).count("1"))                        # 0

0100 1110 -> 1010
Hamming(4, 14) = 2
Hamming(4, 4)  = 0


### 3.4. Đếm số cặp bằng tích: `a * b`

Nếu có `a` đồ vật loại X và `b` đồ vật loại Y, số cặp `(X, Y)` (một từ mỗi loại) là `a * b`. Ví dụ `3` quả táo và `2` quả cam cho `6` cặp táo-cam. Trong bài này: tại một bit, `a` số có bit `1` và `b = n - a` số có bit `0`; mỗi cặp "một số `1`, một số `0`" **khác nhau tại bit ấy**.

In [5]:
tao, cam = ["t1", "t2", "t3"], ["c1", "c2"]
cap = [(x, y) for x in tao for y in cam]
print(len(cap), "=", len(tao), "*", len(cam))        # 6 = 3 * 2
print(cap)

6 = 3 * 2
[('t1', 'c1'), ('t1', 'c2'), ('t2', 'c1'), ('t2', 'c2'), ('t3', 'c1'), ('t3', 'c2')]


### 3.5. Bao nhiêu bit là đủ?

Ràng buộc `nums[i] <= 10^9` và `2^30 = 1.073.741.824 > 10^9`, nên mọi số đều có bit `i` bằng `0` với `i >= 30`. Quét 32 bit (như đề mẫu) hay 30 bit đều đúng; các bit cao chỉ đóng góp `a * (n - a) = 0 * n = 0`.

In [6]:
print((1 << 30) > 10**9, (1 << 29) > 10**9)          # True False  -> cần tối đa 30 bit
print((10**9).bit_length())                           # 30

True False
30


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

**Ví dụ 1:** `[4, 14, 2]`, viết 4 bit:

```
 4 = 0 1 0 0
14 = 1 1 1 0
 2 = 0 0 1 0
```

**Cách 1 (theo cặp):** `d(4,14) = popcount(0100 ^ 1110 = 1010) = 2`; `d(4,2) = popcount(0100 ^ 0010 = 0110) = 2`; `d(14,2) = popcount(1110 ^ 0010 = 1100) = 2`; tổng `6`.

**Cách 2 (theo cột, tức theo từng bit):** nhìn từng **cột** của bảng trên:

| bit | cột (4, 14, 2) | số `1` (`a`) | số `0` (`b`) | `a * b` |
|-----|----------------|--------------|--------------|---------|
| 3 | 0, 1, 0 | 1 | 2 | 2 |
| 2 | 1, 1, 0 | 2 | 1 | 2 |
| 1 | 0, 1, 1 | 2 | 1 | 2 |
| 0 | 0, 0, 0 | 0 | 3 | 0 |

Tổng `2 + 2 + 2 + 0 = 6`. Trùng với cách 1.

**Vì sao cùng đáp án?** Tổng khoảng cách = số (cặp, bit) mà hai số khác nhau tại bit đó. Cách 1 đếm theo cặp rồi cộng các bit; cách 2 đổi **thứ tự cộng**: đếm theo bit rồi cộng các cặp. Đổi thứ tự cộng không đổi tổng. Nhưng cách 2 rẻ hơn rất nhiều vì **số cặp khác nhau tại một bit** tính được bằng công thức `a * b` mà không cần liệt kê cặp.

In [7]:
# Máy hóa phép tính tay: bảng cột cho ví dụ 1
nums = [4, 14, 2]
n = len(nums)
tong = 0
print("bit | cot             | a (so 1) | b (so 0) | a*b")
for i in range(3, -1, -1):
    cot = [x >> i & 1 for x in nums]
    a = sum(cot)
    b = n - a
    tong += a * b
    print(f"{i:3} | {str(cot):15} | {a:8} | {b:8} | {a * b}")
print("tong =", tong)

bit | cot             | a (so 1) | b (so 0) | a*b
  3 | [0, 1, 0]       |        1 |        2 | 2
  2 | [1, 1, 0]       |        2 |        1 | 2
  1 | [0, 1, 1]       |        2 |        1 | 2
  0 | [0, 0, 0]       |        0 |        3 | 0
tong = 6


## 5. Cách 1 — Brute Force / baseline: duyệt mọi cặp

**Ý tưởng:** hai vòng lặp `i < j`; mỗi cặp tính `popcount(nums[i] ^ nums[j])` bằng `bin(...).count("1")` rồi cộng dồn.

In [8]:
def total_hamming_brute(nums):
    n = len(nums)
    tong = 0
    for i in range(n):
        for j in range(i + 1, n):
            tong += bin(nums[i] ^ nums[j]).count("1")
    return tong

for nums, ans in examples:
    print(total_hamming_brute(nums), "(mong doi", ans, ")")

6 (mong doi 6 )
4 (mong doi 4 )


**Độ phức tạp:** Time `O(n^2 * log M)` (`M` là giá trị lớn nhất; mỗi lần đếm bit `O(log M)`), Space `O(1)`.

**Nút thắt:** với `n = 10^4` có khoảng `5 × 10^7` cặp, mỗi cặp chạy `bin(...).count` hàng trăm ns: vài chục giây: **TLE**. Ta thấy thời gian tăng gấp **4** khi `n` tăng gấp **2**:

In [9]:
rng = random.Random(1)
truoc = None
for n in (500, 1000, 2000):
    nums = [rng.randint(0, 10**9) for _ in range(n)]
    t0 = time.perf_counter()
    total_hamming_brute(nums)
    t = time.perf_counter() - t0
    ti_le = f"(x{t / truoc:.1f} so voi truoc)" if truoc else ""
    print(f"n = {n:4}: {t:.3f}s {ti_le}")
    truoc = t

n =  500: 0.039s 


n = 1000: 0.150s (x3.8 so voi truoc)


n = 2000: 0.673s (x4.5 so voi truoc)


## 6. Ý tưởng thuật toán / bước đột phá

**Đổi thứ tự cộng.** Khoảng cách Hamming tổng cộng là số bộ `(cặp, vị trí bit)` mà hai số của cặp khác nhau tại bit đó. Thay vì "với mỗi cặp, đếm các bit khác", ta làm "với mỗi **bit**, đếm các **cặp** khác nhau tại bit đó".

**Đếm cặp tại một bit bằng tích.** Giả sử tại bit `i` có `a` số mang bit `1` và `b = n - a` số mang bit `0`. Một cặp `(x, y)` khác nhau tại bit `i` khi và chỉ khi một số là `1` và số kia là `0`. Có `a * b` cặp như vậy. Tổng: `Σ_{i=0}^{29} a_i * (n - a_i)`.

**Hệ quả:** `O(30 n)` thay vì `O(n^2)`: với `n = 10^4` chỉ cần `3 × 10^5` phép tính.

Ngoài ra còn hai góc nhìn khác của cùng ý tưởng: (i) **chia để trị theo vị trí phần tử**: tổng = tổng(nửa trái) + tổng(nửa phải) + tổng các cặp có một số ở mỗi nửa, và phần cặp chéo cũng tính bằng tích `a * b` theo từng bit; (ii) **trực tuyến**: khi thêm số mới `x`, số cặp "khác nhau tại bit `i`" thêm vào là số các số **đã thấy** có bit `i` khác bit `i` của `x`.

## 7. Các cách tiếp theo

### Cách 2 — Đếm theo từng bit (cách chuẩn)

**Giải thích từng đoạn:**

1. Vòng ngoài `for i in range(30)`: xét bit `i`.
2. `a = sum(x >> i & 1 for x in nums)`: đếm số phần tử có bit `i` bằng `1`.
3. `b = n - a`: số phần tử còn lại.
4. Cộng `a * b` vào đáp án.

In [10]:
def total_hamming_bits(nums):
    n = len(nums)
    ans = 0
    for i in range(30):                         # 10^9 < 2^30
        a = sum(x >> i & 1 for x in nums)       # số phần tử có bit i = 1
        ans += a * (n - a)
    return ans

for nums, ans in examples:
    print(total_hamming_bits(nums), "(mong doi", ans, ")")

6 (mong doi 6 )
4 (mong doi 4 )


**Code nộp LeetCode (Cách 2):**

```python
class Solution:
    def totalHammingDistance(self, nums: List[int]) -> int:
        n = len(nums)
        ans = 0
        for i in range(30):
            a = sum(x >> i & 1 for x in nums)
            ans += a * (n - a)
        return ans
```

**Độ phức tạp:** Time `O(n * 30)`, Space `O(1)`.

### Cách 3 — Chia để trị theo vị trí phần tử: trái + phải + cặp chéo

**Ý tưởng:** chia mảng thành hai nửa theo **chỉ số**. Mọi cặp `(x, y)` thuộc đúng một trong ba loại: cả hai ở nửa trái, cả hai ở nửa phải, hoặc **chéo** (một ở mỗi nửa). Hai loại đầu giải đệ quy; loại chéo tính bằng đếm bit: tại bit `i`, nếu nửa trái có `aL` số bit `1` (trên `nL` số) và nửa phải có `aR` (trên `nR` số) thì số cặp chéo khác nhau tại bit `i` là `aL * (nR - aR) + (nL - aL) * aR`.

**Từng đoạn:**

1. `giai(lo, hi)` xử lý đoạn `nums[lo:hi]` và trả về cặp `(tổng Hamming trong đoạn, mảng đếm bit 1 của đoạn)`.
2. Đoạn một phần tử: tổng `0`, mảng đếm bit là chính các bit của phần tử đó.
3. Ngược lại chia đôi, gọi hai nửa, rồi `cheo = Σ_i aL_i * (nR - aR_i) + (nL - aL_i) * aR_i`.
4. Trả `(tL + tR + cheo, [aL_i + aR_i])`.

In [11]:
def total_hamming_dc(nums):
    def giai(lo, hi):                           # nửa mở [lo, hi)
        if hi - lo == 1:
            return 0, [nums[lo] >> i & 1 for i in range(30)]
        mid = (lo + hi) // 2
        tl, cl = giai(lo, mid)
        tr, cr = giai(mid, hi)
        nl, nr = mid - lo, hi - mid
        cheo = sum(cl[i] * (nr - cr[i]) + (nl - cl[i]) * cr[i] for i in range(30))
        return tl + tr + cheo, [cl[i] + cr[i] for i in range(30)]

    return giai(0, len(nums))[0]

for nums, ans in examples:
    print(total_hamming_dc(nums), "(mong doi", ans, ")")

6 (mong doi 6 )
4 (mong doi 4 )


**Code nộp LeetCode (Cách 3):**

```python
class Solution:
    def totalHammingDistance(self, nums: List[int]) -> int:
        def giai(lo, hi):
            if hi - lo == 1:
                return 0, [nums[lo] >> i & 1 for i in range(30)]
            mid = (lo + hi) // 2
            tl, cl = giai(lo, mid)
            tr, cr = giai(mid, hi)
            nl, nr = mid - lo, hi - mid
            cheo = sum(cl[i] * (nr - cr[i]) + (nl - cl[i]) * cr[i] for i in range(30))
            return tl + tr + cheo, [cl[i] + cr[i] for i in range(30)]

        return giai(0, len(nums))[0]
```

**Độ phức tạp:** Time `O(30 n)` (`n - 1` lần hợp nhất, mỗi lần `O(30)`, cộng `n` lá mỗi lá `O(30)`), Space `O(30 log n)` cho ngăn xếp đệ quy (độ sâu `log n`, mỗi tầng giữ mảng đếm 30 phần tử) cộng mảng đếm ở lá.

**Nhận xét:** về bản chất vẫn là công thức `a * b` của Cách 2, nhưng áp dụng cho **hai nhóm** thay vì cả mảng. Cấu trúc này hữu ích khi bài tổng quát hơn (hợp nhất kết quả của hai nửa, tính song song).

### Cách 4 — Trực tuyến: cộng dồn khi thêm từng số mới

**Ý tưởng:** duyệt từng số `x` theo thứ tự; giữ `ones[i]` = số phần tử đã thấy có bit `i` bằng `1` và `seen` = số phần tử đã thấy. Khi gặp `x`, số cặp mới `(y, x)` với `y` đã thấy, khác nhau tại bit `i`:

- nếu bit `i` của `x` là `1`: những `y` có bit `0`, tức `seen - ones[i]`;
- nếu bit `i` của `x` là `0`: những `y` có bit `1`, tức `ones[i]`.

Cộng vào đáp án rồi cập nhật `ones[i]` và `seen`. Cách này cho thấy bài vẫn giải được **trong một lượt duyệt** (online), hữu ích khi dữ liệu đến theo luồng.

In [12]:
def total_hamming_online(nums):
    ones = [0] * 30
    seen = 0
    ans = 0
    for x in nums:
        for i in range(30):
            bit = x >> i & 1
            ans += (seen - ones[i]) if bit else ones[i]
            ones[i] += bit
        seen += 1
    return ans

for nums, ans in examples:
    print(total_hamming_online(nums), "(mong doi", ans, ")")

6 (mong doi 6 )
4 (mong doi 4 )


**Code nộp LeetCode (Cách 4):**

```python
class Solution:
    def totalHammingDistance(self, nums: List[int]) -> int:
        ones = [0] * 30
        seen = 0
        ans = 0
        for x in nums:
            for i in range(30):
                bit = x >> i & 1
                ans += (seen - ones[i]) if bit else ones[i]
                ones[i] += bit
            seen += 1
        return ans
```

**Độ phức tạp:** Time `O(30 n)`, Space `O(30)` = `O(1)`.

## 8. TRACE từng bước

### 8.1. Cách 2 (đếm theo bit) trên `[4, 14, 2]`

Hàm in từng bit từ cao xuống thấp: cột bit, `a`, `b`, `a * b`, và tổng tích lũy.

In [13]:
def trace_bits(nums, so_bit=4):
    n = len(nums)
    print("nums =", nums, "| nhi phan:", [format(x, f"0{so_bit}b") for x in nums])
    ans = 0
    for i in range(so_bit - 1, -1, -1):
        cot = [x >> i & 1 for x in nums]
        a = sum(cot)
        b = n - a
        ans += a * b
        print(f"bit {i}: cot = {cot}, a = {a}, b = {b}, a*b = {a * b}, tong = {ans}")
    return ans

print("KET QUA:", trace_bits([4, 14, 2]))
print()
print("KET QUA:", trace_bits([4, 14, 4]))

nums = [4, 14, 2] | nhi phan: ['0100', '1110', '0010']
bit 3: cot = [0, 1, 0], a = 1, b = 2, a*b = 2, tong = 2
bit 2: cot = [1, 1, 0], a = 2, b = 1, a*b = 2, tong = 4
bit 1: cot = [0, 1, 1], a = 2, b = 1, a*b = 2, tong = 6
bit 0: cot = [0, 0, 0], a = 0, b = 3, a*b = 0, tong = 6
KET QUA: 6

nums = [4, 14, 4] | nhi phan: ['0100', '1110', '0100']
bit 3: cot = [0, 1, 0], a = 1, b = 2, a*b = 2, tong = 2
bit 2: cot = [1, 1, 1], a = 3, b = 0, a*b = 0, tong = 2
bit 1: cot = [0, 1, 0], a = 1, b = 2, a*b = 2, tong = 4
bit 0: cot = [0, 0, 0], a = 0, b = 3, a*b = 0, tong = 4
KET QUA: 4


### 8.2. Cách 3 (chia để trị theo vị trí) trên `[4, 14, 2]`

Ta in cây đệ quy với 4 bit; thụt dòng thể hiện độ sâu. Mỗi nút cho biết đoạn, tổng nửa trái/phải, số cặp chéo.

In [14]:
def trace_dc(nums, so_bit=4):
    def giai(lo, hi, sau):
        pad = "  " * sau
        if hi - lo == 1:
            c = [nums[lo] >> i & 1 for i in range(so_bit)]
            print(f"{pad}[{lo}:{hi}] = {nums[lo:hi]}: la, bit (thap -> cao) = {c}, tong = 0")
            return 0, c
        mid = (lo + hi) // 2
        print(f"{pad}[{lo}:{hi}] = {nums[lo:hi]}: chia thanh [{lo}:{mid}] va [{mid}:{hi}]")
        tl, cl = giai(lo, mid, sau + 1)
        tr, cr = giai(mid, hi, sau + 1)
        nl, nr = mid - lo, hi - mid
        cheo = sum(cl[i] * (nr - cr[i]) + (nl - cl[i]) * cr[i] for i in range(so_bit))
        print(f"{pad}hop nhat [{lo}:{hi}]: tL = {tl}, tR = {tr}, cap cheo = {cheo} -> tong = {tl + tr + cheo}")
        return tl + tr + cheo, [cl[i] + cr[i] for i in range(so_bit)]
    kq = giai(0, len(nums), 0)[0]
    print("TONG =", kq)
    return kq

trace_dc([4, 14, 2])

[0:3] = [4, 14, 2]: chia thanh [0:1] va [1:3]
  [0:1] = [4]: la, bit (thap -> cao) = [0, 0, 1, 0], tong = 0
  [1:3] = [14, 2]: chia thanh [1:2] va [2:3]
    [1:2] = [14]: la, bit (thap -> cao) = [0, 1, 1, 1], tong = 0
    [2:3] = [2]: la, bit (thap -> cao) = [0, 1, 0, 0], tong = 0
  hop nhat [1:3]: tL = 0, tR = 0, cap cheo = 2 -> tong = 2
hop nhat [0:3]: tL = 0, tR = 2, cap cheo = 4 -> tong = 6
TONG = 6


6

### 8.3. Cách 4 (trực tuyến) trên `[4, 14, 2]`

Với mỗi số mới ta in mức đóng góp (số cặp mới khác nhau) tại từng bit.

In [15]:
def trace_online(nums, so_bit=4):
    ones = [0] * so_bit
    seen = 0
    ans = 0
    for x in nums:
        print(f"them {x} = {format(x, f'0{so_bit}b')} (da thay {seen} so, ones = {ones})")
        them = 0
        for i in range(so_bit - 1, -1, -1):
            bit = x >> i & 1
            d = (seen - ones[i]) if bit else ones[i]
            them += d
            ones[i] += bit
            print(f"    bit {i} = {bit}: cap moi khac = {d}")
        seen += 1
        ans += them
        print(f"  -> cong them {them}, tong = {ans}")
    return ans

print("KET QUA:", trace_online([4, 14, 2]))

them 4 = 0100 (da thay 0 so, ones = [0, 0, 0, 0])
    bit 3 = 0: cap moi khac = 0
    bit 2 = 1: cap moi khac = 0
    bit 1 = 0: cap moi khac = 0
    bit 0 = 0: cap moi khac = 0
  -> cong them 0, tong = 0
them 14 = 1110 (da thay 1 so, ones = [0, 0, 1, 0])
    bit 3 = 1: cap moi khac = 1
    bit 2 = 1: cap moi khac = 0
    bit 1 = 1: cap moi khac = 1
    bit 0 = 0: cap moi khac = 0
  -> cong them 2, tong = 2
them 2 = 0010 (da thay 2 so, ones = [0, 1, 2, 1])
    bit 3 = 0: cap moi khac = 1
    bit 2 = 0: cap moi khac = 2
    bit 1 = 1: cap moi khac = 1
    bit 0 = 0: cap moi khac = 0
  -> cong them 4, tong = 6
KET QUA: 6


## 9. Code nộp LeetCode

Ta nộp **Cách 2** (đếm theo bit): ngắn gọn, dễ chứng minh, `O(30 n)`.

```python
class Solution:
    def totalHammingDistance(self, nums: List[int]) -> int:
        n = len(nums)
        ans = 0
        for i in range(30):
            a = sum(x >> i & 1 for x in nums)
            ans += a * (n - a)
        return ans
```

In [16]:
class Solution:
    def totalHammingDistance(self, nums: List[int]) -> int:
        n = len(nums)
        ans = 0
        for i in range(30):
            a = sum(x >> i & 1 for x in nums)
            ans += a * (n - a)
        return ans

print(Solution().totalHammingDistance([4, 14, 2]))     # 6
print(Solution().totalHammingDistance([4, 14, 4]))     # 4

6
4


## 10. Vì sao thuật toán đúng

**Phân rã theo bit.** Với hai số `x, y` có biểu diễn `x = Σ x_i 2^i`, `y = Σ y_i 2^i` (`x_i, y_i ∈ {0, 1}`), khoảng cách Hamming là `d(x, y) = Σ_i [x_i ≠ y_i]` (dấu ngoặc Iverson bằng `1` nếu đúng). Do đó

```
Σ_{cặp x<y} d(x, y) = Σ_{cặp} Σ_i [x_i ≠ y_i] = Σ_i Σ_{cặp} [x_i ≠ y_i]
```

(đổi thứ tự hai dấu tổng hữu hạn). Với mỗi `i`, `Σ_{cặp} [x_i ≠ y_i]` là số cặp không có thứ tự gồm một phần tử có bit `1` và một phần tử có bit `0`, bằng `a_i * (n - a_i)`: mỗi cặp chọn **một** phần tử từ `a_i` phần tử bit `1` và **một** từ `n - a_i` phần tử bit `0`, và mỗi cặp không có thứ tự như vậy được đếm đúng một lần.

**Vì sao các bit cao (`i >= 30`) không cần?** Với `nums[i] <= 10^9 < 2^30` thì `a_i = 0`, đóng góp `0`.

**Cách 3 (chia để trị theo vị trí).** Mỗi cặp `(x, y)` với `x` đứng trước `y` trong mảng thuộc đúng một trong ba loại: cả hai thuộc nửa trái, cả hai thuộc nửa phải, hoặc một ở mỗi nửa. Hai loại đầu được tính bởi đệ quy (quy nạp theo độ dài đoạn); loại chéo, theo phân rã theo bit ở trên, bằng `Σ_i` (số cặp chéo khác nhau tại bit `i`) `= Σ_i aL_i * (nR - aR_i) + (nL - aL_i) * aR_i`, vì cặp chéo khác nhau tại bit `i` khi số trái là `1` và số phải là `0` (`aL_i * (nR - aR_i)` cặp) hoặc ngược lại (`(nL - aL_i) * aR_i` cặp), và hai trường hợp này loại trừ nhau.

**Cách 4 (trực tuyến).** Mỗi cặp `(y, x)` với `y` đến trước `x` được đếm đúng một lần, vào lúc `x` đến, theo từng bit khác nhau: nếu bit `i` của `x` là `1` thì các `y` khác `x` tại bit đó là các `y` có bit `0`, tức `seen - ones[i]`; nếu là `0` thì là các `y` có bit `1`, tức `ones[i]`.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|------|------|-------|---------|
| 1. Vét cạn mọi cặp | `O(n^2 log M)` | `O(1)` | oracle, TLE khi `n = 10^4` |
| 2. Đếm theo bit (nộp) | `O(n log M)` | `O(1)` | `log M = 30` |
| 3. Chia để trị theo vị trí | `O(n log M)` | `O(log M log n)` | chỉ để học cấu trúc |
| 4. Trực tuyến | `O(n log M)` | `O(log M)` | một lượt duyệt, hợp dữ liệu luồng |

Với `n = 10^4`: Cách 1 cỡ `5 × 10^7` cặp; Cách 2 cỡ `3 × 10^5` thao tác. Đáp án tối đa `30 * (n/2)^2 = 30 * 2.5 × 10^7 = 7.5 × 10^8 < 2^31`, đúng cam kết "vừa 32 bit" của đề.

## 12. Bộ test

Cách 1 (vét cạn) là oracle độc lập; đối chiếu Cách 2, 3, 4 và `Solution`.

In [17]:
cac_cach = {
    "bits":     total_hamming_bits,
    "dc":       total_hamming_dc,
    "online":   total_hamming_online,
    "Solution": lambda nums: Solution().totalHammingDistance(list(nums)),
}

def so_het(nums, oracle=None):
    if oracle is None:
        oracle = total_hamming_brute(nums)
    for ten, f in cac_cach.items():
        v = f(list(nums))
        assert v == oracle, (ten, nums, v, oracle)
    return oracle

for nums, ans in examples:
    assert so_het(nums, ans) == ans
print("Vi du de OK")

Vi du de OK


In [18]:
# Edge case
tests = [
    ([0], 0),                                    # một phần tử: không có cặp
    ([5], 0),
    ([0, 0], 0),                                 # hai số 0
    ([7, 7, 7, 7], 0),                           # tất cả bằng nhau
    ([0, 1], 1),
    ([0, 3], 2),
    ([0, 10**9], bin(10**9).count("1")),         # giá trị lớn nhất
    ([2**29, 0, 2**29, 0], 4),                   # chỉ bit 29 khác: 2 số 1, 2 số 0 -> 2*2
    ([1, 2, 4, 8], 12),                          # mỗi số một bit khác nhau: C(4,2) cặp * 2 bit = 12
    ([0, 1, 2, 3], 8),                           # 0..3: tổng Hamming
    ([4, 14, 2, 4, 14], None),
    (list(range(16)), None),                     # đủ mọi giá trị 4 bit
]
for nums, ans in tests:
    kq = so_het(nums)
    if ans is not None:
        assert kq == ans, (nums, kq, ans)
print("Edge case OK:", len(tests), "test")
print("tong Hamming cua 0..15 =", total_hamming_brute(list(range(16))), "(= 4 bit * 8 * 8 = 256)")
assert total_hamming_brute(list(range(16))) == 256

Edge case OK: 12 test
tong Hamming cua 0..15 = 256 (= 4 bit * 8 * 8 = 256)


**Phản ví dụ** cho các cách hiểu sai: `[1, 2, 4, 8]` có đáp án `12`, **không phải** `4` (số bit) hay `6` (số cặp): mỗi cặp khác nhau tại đúng `2` bit.

In [19]:
# Đối chiếu ngẫu nhiên có seed: n nhỏ, nhiều kiểu miền giá trị
rng = random.Random(477)
for _ in range(1500):
    n = rng.randint(1, 12)
    hi = rng.choice([1, 3, 7, 15, 255, 10**9])
    nums = [rng.randint(0, hi) for _ in range(n)]
    so_het(nums)
print("1500 test ngau nhien (nho) khop voi vet can")

# n cỡ vừa: vét cạn vẫn chạy
for _ in range(10):
    n = rng.randint(200, 800)
    nums = [rng.randint(0, 10**9) for _ in range(n)]
    so_het(nums)
print("10 test n tới 800 khop voi vet can")

1500 test ngau nhien (nho) khop voi vet can


10 test n tới 800 khop voi vet can


In [20]:
# Ca lớn nhất theo ràng buộc: n = 10^4, so các cách nhanh với nhau
rng = random.Random(4770)
nums = [rng.randint(0, 10**9) for _ in range(10**4)]
kq = {}
for ten, f in cac_cach.items():
    t0 = time.perf_counter()
    kq[ten] = f(list(nums))
    print(f"{ten:9}: {kq[ten]}  ({time.perf_counter() - t0:.3f}s)")
assert len(set(kq.values())) == 1
assert kq["bits"] < 2**31

# Ca xấu cho đáp án lớn: nửa số toàn bit 1 (2^30 - 1), nửa số bằng 0 -> a = n/2 ở mọi bit
nums = [2**30 - 1] * 5000 + [0] * 5000
ans = Solution().totalHammingDistance(nums)
print("nua 2^30-1, nua 0:", ans, "= 30 * 5000 * 5000 =", 30 * 5000 * 5000)
assert ans == 30 * 5000 * 5000 and ans < 2**31
assert total_hamming_online(nums) == total_hamming_dc(nums) == ans

bits     : 749373203  (0.014s)
dc       : 749373203  (0.074s)
online   : 749373203  (0.052s)
Solution : 749373203  (0.026s)
nua 2^30-1, nua 0: 750000000 = 30 * 5000 * 5000 = 750000000


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Cộng giá trị XOR thay vì đếm bit

`a ^ b` là một **số**; tổng các giá trị XOR hoàn toàn khác tổng số bit `1`.

In [21]:
def sai_cong_xor(nums):
    n = len(nums)
    return sum(nums[i] ^ nums[j] for i in range(n) for j in range(i + 1, n))     # SAI: thiếu đếm bit

print("sai :", sai_cong_xor([4, 14, 2]), "| dung:", Solution().totalHammingDistance([4, 14, 2]))

sai : 28 | dung: 6


**Vì sao sai:** `4 ^ 14 = 10`, `4 ^ 2 = 6`, `14 ^ 2 = 12`, tổng `28`; khoảng cách Hamming là số bit `1` của mỗi giá trị đó (`2 + 2 + 2`).

**Code đúng:** `bin(a ^ b).count("1")`.

### Lỗi 2 — Cộng `a` hoặc `a + b` thay vì `a * b`

Một số bạn đếm "số bit 1" thay vì số **cặp** khác nhau.

In [22]:
def sai_cong_a(nums):
    ans = 0
    for i in range(30):
        a = sum(x >> i & 1 for x in nums)
        ans += a                                       # SAI: phải là a * (n - a)
    return ans

def sai_a_nhan_a(nums):
    n = len(nums)
    ans = 0
    for i in range(30):
        a = sum(x >> i & 1 for x in nums)
        ans += a * a                                   # SAI: cặp (1,1) không khác nhau
    return ans

for nums in ([4, 14, 2], [1, 2, 4, 8]):
    print(nums, "| sai a:", sai_cong_a(nums), "| sai a*a:", sai_a_nhan_a(nums), "| dung:", Solution().totalHammingDistance(nums))

[4, 14, 2] | sai a: 5 | sai a*a: 9 | dung: 6
[1, 2, 4, 8] | sai a: 4 | sai a*a: 4 | dung: 12


**Vì sao sai:** cặp chỉ **khác nhau** khi một số `1` và một số `0`; số cặp như vậy là `a * (n - a)`, không phải `a` hay `a * a`.

**Code đúng:** `ans += a * (n - a)`.

### Lỗi 3 — Đếm mỗi cặp hai lần (cặp có thứ tự)

Nếu tính `a * b * 2` hoặc duyệt hai vòng `i`, `j` cả hai chiều trong vét cạn, kết quả gấp đôi.

In [23]:
def sai_dem_hai_lan(nums):
    n = len(nums)
    tong = 0
    for i in range(n):
        for j in range(n):                             # SAI: j chạy cả i < j và i > j
            tong += bin(nums[i] ^ nums[j]).count("1")
    return tong

print("sai :", sai_dem_hai_lan([4, 14, 2]), "| dung:", Solution().totalHammingDistance([4, 14, 2]))

sai : 12 | dung: 6


**Vì sao sai:** mỗi cặp không thứ tự `{x, y}` bị cộng cả `(x, y)` lẫn `(y, x)`. **Code đúng:** `for j in range(i + 1, n)`.

### Lỗi 4 — Quét thiếu số bit

Nếu chỉ quét `range(8)` hay `range(16)`, các số lớn mất các bit cao.

In [24]:
def sai_it_bit(nums, so_bit):
    n = len(nums)
    ans = 0
    for i in range(so_bit):                            # SAI nếu so_bit < 30
        a = sum(x >> i & 1 for x in nums)
        ans += a * (n - a)
    return ans

nums = [10**9, 0, 2**20]
for so_bit in (8, 16, 30):
    print(f"quet {so_bit:2} bit ->", sai_it_bit(nums, so_bit))
print("dung:", Solution().totalHammingDistance(nums))

quet  8 bit -> 0
quet 16 bit -> 8
quet 30 bit -> 26
dung: 26


**Vì sao sai:** `10^9` cần `30` bit; cắt ở `8` hay `16` bit làm mất đóng góp bit cao. **Code đúng:** `range(30)` (hoặc `range(32)` cho an toàn).

### Lỗi 5 — Dùng `x & (1 << i)` rồi cộng như thể là `0/1`

`x & (1 << i)` cho `0` hoặc `2^i`, **không** phải `0/1`. Cộng chúng cho tổng sai.

In [25]:
def sai_mask(nums):
    n = len(nums)
    ans = 0
    for i in range(30):
        a = sum(x & (1 << i) for x in nums)            # SAI: mỗi số cộng 2^i chứ không phải 1
        ans += a * (n - a)
    return ans

nums = [4, 14, 2]
print("sai :", sai_mask(nums), "| dung:", Solution().totalHammingDistance(nums))

# Sửa: ép về bool/0-1 bằng dịch bit hoặc so sánh
def dung_mask(nums):
    n = len(nums)
    ans = 0
    for i in range(30):
        a = sum(1 for x in nums if x & (1 << i))        # đếm các số có bit i
        ans += a * (n - a)
    return ans
print("dung (so sanh):", dung_mask(nums))

sai : -84 | dung: 6
dung (so sanh): 6


**Vì sao sai:** `a` phải là **số lượng** phần tử có bit `1`, không phải tổng các giá trị mặt nạ.

### Lỗi 6 — "Chia để trị theo bit" chỉ đếm mỗi cặp một lần

Ý tưởng nghe rất hợp lý: tách mảng thành nhóm bit `0` và nhóm bit `1` tại bit cao nhất, cộng `len(zeros) * len(ones)`, rồi đệ quy riêng trong từng nhóm với bit thấp hơn. Lỗi tư duy: hai số đã bị tách ở bit cao sẽ **không bao giờ gặp nhau** ở các tầng thấp, nên chỉ được đếm **một lần** (tại bit cao nhất mà chúng khác nhau), dù chúng có thể khác nhau ở nhiều bit.

In [26]:
def sai_chia_nhom_theo_bit(nums):
    def giai(mang, bit):
        if bit < 0 or len(mang) <= 1:
            return 0
        zeros = [x for x in mang if not x >> bit & 1]
        ones = [x for x in mang if x >> bit & 1]
        return len(zeros) * len(ones) + giai(zeros, bit - 1) + giai(ones, bit - 1)   # SAI
    return giai(list(nums), 29)

for nums in ([4, 14, 2], [1, 2, 4, 8], [0, 3]):
    print(nums, "| sai:", sai_chia_nhom_theo_bit(nums), "| dung:", Solution().totalHammingDistance(nums))

[4, 14, 2] | sai: 3 | dung: 6
[1, 2, 4, 8] | sai: 6 | dung: 12
[0, 3] | sai: 1 | dung: 2


**Vì sao sai:** `sai_chia_nhom_theo_bit` thực chất đếm **số cặp khác nhau** (mỗi cặp một lần), không phải **tổng số bit khác nhau**. Với `[0, 3]` hai số khác nhau ở **hai** bit nhưng chỉ được tính `1`.

**Code đúng:** đếm theo bit trên **toàn mảng** (Cách 2), hoặc chia để trị theo vị trí phần tử có đếm cặp chéo theo từng bit (Cách 3).

### Lỗi 7 — Vét cạn `O(n^2)` với `n = 10^4`

Đúng về kết quả nhưng quá chậm (mục 5): gấp đôi `n` thì thời gian gấp bốn. Luôn kiểm tra ràng buộc `n` trước khi chọn thuật toán.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "tổng (hoặc đếm) trên **mọi cặp** của một đại lượng **cộng được theo từng bit**" → tách theo bit, mỗi bit tính bằng tích hai nhóm.
- **Mẫu chung:** `Σ_{cặp} f(x, y)` với `f` = tổng các hàm theo từng bit → đổi thứ tự tổng, đếm theo bit rồi nhân `a * (n - a)`.
- **Trực tuyến:** nếu dữ liệu đến theo luồng, duy trì `ones[i]` và cộng đóng góp khi có số mới (Cách 4).
- **Kiểm tra bằng vét cạn nhỏ** trước khi tin một "chia để trị" nghe có lý (mục 10 là ví dụ).
- **Giới hạn bit:** nhìn ràng buộc: `10^9 < 2^30` → 30 bit; 32 bit cho an toàn.
- **Bài họ hàng:** LeetCode 461 *Hamming Distance*, 1863 *Sum of All Subset XOR Totals*, 2275 *Largest Combination With Bitwise AND Greater Than Zero*, 1442 *Count Triplets That Can Form Two Arrays of Equal XOR*, 2425 *Bitwise XOR of All Pairings*.

## 15. Biến thể

1. **Tổng khoảng cách Hamming giữa hai mảng** `A`, `B` (mọi cặp một từ `A`, một từ `B`): tại mỗi bit `a_A * (|B| - a_B) + (|A| - a_A) * a_B`.
2. **Tổng XOR của mọi cặp**: vẫn tách theo bit, nhưng nhân thêm `2^i`: `Σ_i 2^i * a_i * (n - a_i)`.
3. **Tổng AND (hoặc OR) của mọi cặp:** AND: `C(a_i, 2) * 2^i`; OR: `(C(n, 2) - C(n - a_i, 2)) * 2^i`.
4. **Khoảng cách Hamming nhỏ nhất/lớn nhất giữa hai số bất kỳ**: cần cấu trúc khác (Trie nhị phân), không phải đếm theo bit.
5. **Số có độ rộng lớn hơn (ví dụ `10^18`)**: tăng số bit quét lên `60`.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|------|---------|------|-------|--------------|
| 1. Vét cạn mọi cặp | XOR + đếm bit từng cặp | `O(n^2 log M)` | `O(1)` | oracle, `n` nhỏ |
| 2. Đếm theo bit | `Σ a (n - a)` | `O(n log M)` | `O(1)` | **nộp bài**, đơn giản nhất |
| 3. Chia để trị theo vị trí | trái + phải + cặp chéo | `O(n log M)` | `O(log M log n)` | khi cần cấu trúc đệ quy/song song |
| 4. Trực tuyến | cộng dồn khi số mới đến | `O(n log M)` | `O(log M)` | dữ liệu luồng |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `bin(10).count("1")`; (b) `5 >> 1 & 1`; (c) `6 ^ 3`; (d) `Solution().totalHammingDistance([1, 3])`; (e) `Solution().totalHammingDistance([5, 5, 5])`.

**Bài 2 (tính tay).** Với `nums = [1, 2, 3]` (viết 2 bit): lập bảng cột, tính `a`, `b`, `a * b` mỗi bit, rồi tổng. Đối chiếu với cách tính theo cặp.

**Bài 3 (điền TODO).** Hoàn thành `dem_bit_1(nums, i)` đếm số phần tử có bit `i` bằng `1`, và `dong_gop(nums, i)` trả về `a * (n - a)` của bit `i`.

In [27]:
def dem_bit_1(nums, i):
    # TODO: trả về số phần tử x trong nums có x >> i & 1 == 1
    return None

def dong_gop(nums, i):
    # TODO: a = dem_bit_1(nums, i); trả về a * (len(nums) - a)
    return None

# KHÔNG gọi dem_bit_1 / dong_gop ở đây để "Run All" không lỗi (chưa điền xong).

**Bài 4 (tự viết lại).** Viết `total_hamming_todo(nums)` bằng Cách 4 (trực tuyến) **không nhìn lại** mục 7. Kiểm tra bằng hai ví dụ của đề và với vét cạn.

**Bài 5 (biến thể / mini challenge).** Viết `total_xor_pairs(nums)` trả về **tổng giá trị XOR** của mọi cặp (không phải số bit). Kiểm tra: `total_xor_pairs([4, 14, 2]) == 28` (`10 + 6 + 12`). Gợi ý: nhân đóng góp của bit `i` với `2^i`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: (a) `2` (`1010`); (b) `5 = 101`, dịch phải 1 được `10`, `& 1` = `0`; (c) `5` (`110 ^ 011 = 101`); (d) `1` (`01` và `11` khác ở bit 1); (e) `0`.
- **Mức 2 (vừa):** Bài 2: `1 = 01`, `2 = 10`, `3 = 11`. Bit 1: cột `0,1,1` → `a = 2`, `b = 1` → `2`. Bit 0: cột `1,0,1` → `a = 2`, `b = 1` → `2`. Tổng `4`. Theo cặp: `d(1,2) = 2`, `d(1,3) = 1`, `d(2,3) = 1` → `4`. Bài 4: nhớ cập nhật `ones[i]` **sau** khi cộng đóng góp.
- **Mức 3 (sâu):** Bài 5: tổng XOR của mọi cặp `= Σ_i 2^i * a_i * (n - a_i)`; kiểm tra: `4=0100, 14=1110, 2=0010`: bit 3 → `2^3 * 2 = 16`; bit 2 → `2^2 * 2 = 8`; bit 1 → `2^1 * 2 = 4`; bit 0 → `0`; tổng `28`.

## 19. Đáp án tham khảo cho bài tập

In [28]:
# Bài 1
assert bin(10).count("1") == 2
assert (5 >> 1 & 1) == 0
assert (6 ^ 3) == 5
assert Solution().totalHammingDistance([1, 3]) == 1
assert Solution().totalHammingDistance([5, 5, 5]) == 0
print("Bài 1 OK")

# Bài 2
nums = [1, 2, 3]
tong = 0
for i in (1, 0):
    cot = [x >> i & 1 for x in nums]
    a = sum(cot)
    print(f"Bài 2: bit {i}: cot {cot}, a = {a}, b = {len(nums) - a}, a*b = {a * (len(nums) - a)}")
    tong += a * (len(nums) - a)
assert tong == 4 == total_hamming_brute(nums) == Solution().totalHammingDistance(nums)
print("Bài 2 OK")

# Bài 3
def dem_bit_1_ans(nums, i):
    return sum(x >> i & 1 for x in nums)

def dong_gop_ans(nums, i):
    a = dem_bit_1_ans(nums, i)
    return a * (len(nums) - a)

assert dem_bit_1_ans([4, 14, 2], 3) == 1 and dem_bit_1_ans([4, 14, 2], 0) == 0
assert [dong_gop_ans([4, 14, 2], i) for i in range(4)] == [0, 2, 2, 2]
assert sum(dong_gop_ans([4, 14, 2], i) for i in range(30)) == 6
print("Bài 3 OK")

# Bài 4
def total_hamming_todo_ans(nums):
    ones = [0] * 30
    seen = 0
    ans = 0
    for x in nums:
        for i in range(30):
            bit = x >> i & 1
            ans += (seen - ones[i]) if bit else ones[i]
            ones[i] += bit
        seen += 1
    return ans

for nums, ans in examples:
    assert total_hamming_todo_ans(nums) == ans
random.seed(47)
for _ in range(300):
    nums = [random.randint(0, 10**9) for _ in range(random.randint(1, 15))]
    assert total_hamming_todo_ans(nums) == total_hamming_brute(nums)
print("Bài 4 OK")

# Bài 5
def total_xor_pairs(nums):
    n = len(nums)
    ans = 0
    for i in range(30):
        a = sum(x >> i & 1 for x in nums)
        ans += (1 << i) * a * (n - a)
    return ans

def total_xor_brute(nums):
    return sum(nums[i] ^ nums[j] for i in range(len(nums)) for j in range(i + 1, len(nums)))

assert total_xor_pairs([4, 14, 2]) == 28
random.seed(48)
for _ in range(300):
    nums = [random.randint(0, 10**9) for _ in range(random.randint(1, 15))]
    assert total_xor_pairs(nums) == total_xor_brute(nums)
print("Bài 5 OK")

Bài 1 OK
Bài 2: bit 1: cot [0, 1, 1], a = 2, b = 1, a*b = 2
Bài 2: bit 0: cot [1, 0, 1], a = 2, b = 1, a*b = 2
Bài 2 OK
Bài 3 OK
Bài 4 OK
Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template (tổng trên mọi cặp của đại lượng cộng được theo từng bit):**

```python
n = len(nums)
ans = 0
for i in range(30):                    # 10^9 < 2^30
    a = sum(x >> i & 1 for x in nums)  # số phần tử có bit i = 1
    ans += a * (n - a)                 # số cặp khác nhau tại bit i
return ans
```

**Checklist:**

- [ ] Đã đổi thứ tự cộng: **theo bit** rồi mới **theo cặp** (`Σ_bit Σ_cặp`).
- [ ] Tại mỗi bit: đóng góp là `a * (n - a)` (không phải `a`, `a * a`, hay `2 * a * b`).
- [ ] Số bit quét đủ (`30` cho `10^9`; `32` an toàn), không cắt ngắn.
- [ ] `a` là **số lượng** phần tử có bit `1` (dùng `x >> i & 1`, không dùng tổng của `x & (1 << i)`).
- [ ] Không dùng vét cạn `O(n^2)` khi `n = 10^4`.
- [ ] Mọi "chia để trị" tự nghĩ ra đều đã **đối chiếu với vét cạn** trên input nhỏ.
- [ ] Thử tay `[4,14,2]` (6), `[4,14,4]` (4), `[1,2,4,8]` (12), `[0]` (0).
- [ ] Đúng signature `class Solution` / `totalHammingDistance(self, nums: List[int]) -> int`; độ phức tạp `O(30 n)` thời gian, `O(1)` bộ nhớ.